# backward-func-lookup — ex3: alias one back_fn under multiple (fwd,argnum) keys

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-func-lookup`. Running the final beacon cell reports progress against the `Backprop: BackwardFuncLookup` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: BackwardFuncLookup` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-func-lookup`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-func-lookup"
DD_SUBTOPIC = "Backprop: BackwardFuncLookup"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BackwardFuncLookup — aliasing one back_fn under multiple keys

Ex1 built the class; ex2 dispatched it across a 2-op reverse pass. The deepening move exploits a property the dispatcher REQUIRES but neither earlier ex tested: the SAME `back_fn` object can be registered under multiple `(fwd, argnum)` keys.

**Where this comes up.** Symmetric binary ops — `add`, `multiply` — have identical (up to operand order) back-fns at both argnums:

```
add_back(grad_out, out, x, y) = grad_out          # same for argnum 0 AND argnum 1
BFL.add_back_func(t.add, 0, add_back)
BFL.add_back_func(t.add, 1, add_back)             # same function object both times
```

**Both lookups must return THE SAME object** (same `id`), not a clone. This is what lets the dispatcher route either argnum to the shared implementation without paying for duplicate registration storage or risking divergence between two copies. ARENA's actual back-fn table relies on this — most single-function entries are also aliased across argnums for the symmetric ops.

### Exercise 3 — alias one back_fn under multiple (fwd,argnum) keys

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the BackwardFuncLookup contract that the same back_fn object can be registered under multiple (fwd, argnum) keys, and verify both lookups return THE SAME object (identity, not equality).
> Keywords: backward-func-lookup, alias, symmetric-op, shared-fn
> ```

**KCs targeted:** `shared-back-fn-aliasing`, `lookup-returns-same-object`

Implement `BackwardFuncLookup` (same `add_back_func` + `get_back_func` API as ex1) AND a registration helper `register_symmetric(BFL, fwd_fn, back_fn)` that aliases the SINGLE `back_fn` under BOTH `argnum=0` and `argnum=1` keys.

Requirements:

1. `BackwardFuncLookup.__init__()` — initialise an empty dict.
2. `add_back_func(forward_fn, argnum, back_fn)` — store at `(forward_fn, argnum)`.
3. `get_back_func(forward_fn, argnum)` — return the stored fn, raise `KeyError` if missing.
4. `register_symmetric(BFL, fwd_fn, back_fn)` — call `add_back_func` TWICE with the SAME `back_fn` object (argnum 0, argnum 1).

The test then verifies that `BFL.get_back_func(t.add, 0)` and `BFL.get_back_func(t.add, 1)` return the IDENTICAL function object (`is` check, not just `==`). It also verifies the symmetric pattern for `t.multiply` — `multiply_back` aliased under both argnums — and confirms updating the shared fn affects both lookups together (showing they're truly the same object).

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, argnum, back_fn):
        self.back_funcs[(forward_fn, argnum)] = back_fn

    def get_back_func(self, forward_fn, argnum):
        key = (forward_fn, argnum)
        if key not in self.back_funcs:
            raise KeyError(
                f'no back-fn registered for fwd={getattr(forward_fn, "__name__", forward_fn)!r} argnum={argnum}'
            )
        return self.back_funcs[key]


def ex3_register_symmetric(BFL, fwd_fn, back_fn):
    BFL.add_back_func(fwd_fn, 0, back_fn)
    BFL.add_back_func(fwd_fn, 1, back_fn)
    return BFL

register_symmetric = ex3_register_symmetric


<details><summary>Solution</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, argnum, back_fn):
        self.back_funcs[(forward_fn, argnum)] = back_fn

    def get_back_func(self, forward_fn, argnum):
        key = (forward_fn, argnum)
        if key not in self.back_funcs:
            raise KeyError(
                f'no back-fn registered for fwd={getattr(forward_fn, "__name__", forward_fn)!r} argnum={argnum}'
            )
        return self.back_funcs[key]


def ex3_register_symmetric(BFL, fwd_fn, back_fn):
    BFL.add_back_func(fwd_fn, 0, back_fn)
    BFL.add_back_func(fwd_fn, 1, back_fn)
    return BFL

register_symmetric = ex3_register_symmetric
```

**`is` not `==`.** The test uses `is` because aliasing is about IDENTITY: the dispatcher must dispatch to one function object, not two. If a user (or a buggy register helper) accidentally registered `copy.copy(back_fn)` under one argnum, `==` would still hold for function objects but `is` would fail — exactly what we want to catch.

**Why aliasing instead of a single `add_symmetric(fwd, back_fn)` method.** Two flat keys keep dispatch O(1) and uniform — the reverse pass never has to check 'is this op symmetric?'. The asymmetry handling lives only at REGISTRATION time, not at LOOKUP time.

**Same trick for `t.add`, `t.multiply`, bitwise `t.bitwise_and`, etc.** Anywhere both back-fns are identical (or can be expressed as one fn with argnum-agnostic logic), aliasing saves an entry and a divergence risk.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()